# Travel Planner Generator

This notebook implements an AI-powered meta-software development workflow for the DTS114TC coursework. It turns a travel planning business problem into software artefacts, including SDLC documentation, UML diagrams, a Flask API, a travel planning website, and deployment assets.

## Environment Requirements

To run this notebook, use a Python/Jupyter environment with `requests`, `python-dotenv`, `flask`, `flask-cors`, and `pytest` installed. An APIFree API key should be provided through a `.env` file or environment variable as `APIFREE_API_KEY`. Internet access is required for APIFree and PlantUML diagram rendering. Docker Desktop is only required for the deployment step in Task2.

## Development Methodology

The project follows an AI-driven development lifecycle supported by Agile-style iteration. The workflow is organised into three phases: Inception, Construction, and Operation.

## Setup

This section prepares imports, output folders, and APIFree API configuration. The notebook requires an APIFree API key for AI generation.

In [ ]:
from pathlib import Path
import json
import os

PROJECT_ROOT = Path.cwd().parent
TASK2_DIR = PROJECT_ROOT / "Task2"
APP_DIR = TASK2_DIR / "app"
DOCS_DIR = TASK2_DIR / "docs"
UML_DIR = TASK2_DIR / "uml"
ASSETS_DIR = APP_DIR / "static" / "images"

for directory in [APP_DIR, DOCS_DIR, UML_DIR, ASSETS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    print("python-dotenv is not installed. The notebook will use existing environment variables only.")

API_KEY = os.getenv("APIFREE_API_KEY")
APIFREE_BASE_URL = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1")
APIFREE_MODEL = os.getenv("APIFREE_MODEL", "gpt-5-mini")
APIFREE_IMAGE_MODEL = os.getenv("APIFREE_IMAGE_MODEL", "google/nano-banana-2")

print("APIFree API key detected" if API_KEY else "APIFREE_API_KEY is not configured yet")
print(f"Text model: {APIFREE_MODEL}")
print(f"Image model: {APIFREE_IMAGE_MODEL}")

# Phase 1: Inception

Inception converts the business problem into software development context, including a problem statement, personas, requirements, and user stories.

In [ ]:
business_problem = """
A small travel agency needs a web application that helps tourists generate personalised city travel itineraries. Users should be able to enter a destination, trip length, budget level, and interests. The system should return a clear day-by-day itinerary and present it through both a Flask API and a website.
""".strip()

project_context = {
    "project_name": "Travel Planner Generator",
    "business_problem": business_problem,
    "demo_destination": "Kyoto",
    "demo_days": 3,
    "demo_budget": "medium",
    "demo_interests": ["culture", "food", "nature"],
}

print(project_context["business_problem"])

## Generated SDLC Documentation

This section uses the APIFree API to create the problem statement, personas, requirements, PRD, and user stories.

In [ ]:
def generate_text(task_name, prompt, context):
    """Generate text with the APIFree chat endpoint."""
    api_key = os.getenv("APIFREE_API_KEY")
    base_url = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1")
    model = os.getenv("APIFREE_MODEL", "gpt-5-mini")

    if not api_key:
        raise RuntimeError("APIFREE_API_KEY is not configured. Add it to the environment before running this notebook.")

    import requests

    response = requests.post(
        f"{base_url.rstrip('/')}/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a software engineering assistant producing concise coursework artefacts."},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.3,
        },
        timeout=180,
    )
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        message = data["error"].get("message", data["error"])
        raise RuntimeError(f"APIFree generation failed for {task_name}: {message}")
    if "choices" not in data:
        raise RuntimeError(f"APIFree response for {task_name} did not include choices: {data}")
    return data["choices"][0]["message"]["content"].strip()

In [ ]:
problem_statement_prompt = f"""
Given the business problem below, write one concise software engineering problem statement.

Business problem:
{business_problem}
""".strip()

personas_prompt = f"""
Generate 3-4 concise user personas for this software project.
Each persona should include the user's role, goal, and main concern.

Business problem:
{business_problem}
""".strip()

requirements_prompt = f"""
Write functional and non-functional requirements for this project in markdown.
Keep the scope suitable for a coursework Flask API and website.

Business problem:
{business_problem}
""".strip()

prd_prompt = f"""
Write a concise Product Requirements Document for this project with headings:
Overview, Goals, Scope, Constraints, Success Criteria.

Business problem:
{business_problem}
""".strip()

user_stories_prompt = f"""
Return only valid JSON for 3-5 Agile user stories using this schema:
{{
  "user_stories": [
    {{
      "id": "US1",
      "role": "...",
      "goal": "...",
      "benefit": "...",
      "acceptance_criteria": ["...", "..."]
    }}
  ]
}}

Business problem:
{business_problem}
""".strip()

generated_docs = {
    "problem_statement": generate_text("problem_statement", problem_statement_prompt, project_context),
    "personas": generate_text("personas", personas_prompt, project_context),
    "requirements": generate_text("requirements", requirements_prompt, project_context),
    "prd": generate_text("prd", prd_prompt, project_context),
    "user_stories": generate_text("user_stories", user_stories_prompt, project_context),
}

for name, content in generated_docs.items():
    suffix = "json" if name == "user_stories" else "md"
    output_path = DOCS_DIR / f"{name}.{suffix}"
    output_path.write_text(content, encoding="utf-8")
    print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")

In [ ]:
print("Problem statement:\n")
print(generated_docs["problem_statement"])

print("\nUser stories preview:\n")
print(generated_docs["user_stories"][:800])

# Phase 2: Construction

Construction will generate UML diagrams, Flask API code, website code, and an automatically generated destination image.

## Generated UML and Application Code

This section generates UML diagrams from the SDLC documentation and renders them through the public PlantUML server.

In [ ]:
import re
import zlib
from urllib.parse import quote


def extract_plantuml(text):
    """Extract PlantUML code from a model response."""
    fenced_match = re.search(r"```(?:plantuml|puml)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if fenced_match:
        text = fenced_match.group(1).strip()
    start = text.find("@startuml")
    end = text.find("@enduml")
    if start == -1 or end == -1:
        raise ValueError("The generated UML response does not contain @startuml and @enduml.")
    return text[start:end + len("@enduml")].strip()


def plantuml_encode(text):
    """Encode PlantUML text for the PlantUML server."""
    data = zlib.compress(text.encode("utf-8"))[2:-4]
    alphabet = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-_"

    def encode_3bytes(bytes_chunk):
        if len(bytes_chunk) < 3:
            bytes_chunk += b"\x00" * (3 - len(bytes_chunk))
        b1, b2, b3 = bytes_chunk
        c1 = b1 >> 2
        c2 = ((b1 & 0x3) << 4) | (b2 >> 4)
        c3 = ((b2 & 0xF) << 2) | (b3 >> 6)
        c4 = b3 & 0x3F
        return alphabet[c1] + alphabet[c2] + alphabet[c3] + alphabet[c4]

    return "".join(encode_3bytes(data[i:i + 3]) for i in range(0, len(data), 3))


def render_plantuml_png(plantuml_text, output_path):
    """Render a PlantUML diagram to PNG using the public PlantUML server."""
    import requests

    encoded = plantuml_encode(plantuml_text)
    url = f"https://www.plantuml.com/plantuml/png/{encoded}"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    output_path.write_bytes(response.content)
    return url

In [ ]:
uml_context = "\n\n".join([
    "Business Problem:\n" + business_problem,
    "Problem Statement:\n" + generated_docs["problem_statement"],
    "Requirements:\n" + generated_docs["requirements"],
    "User Stories:\n" + generated_docs["user_stories"],
])

uml_prompts = {
    "use_case_diagram": f"""
Generate a UML use case diagram in valid PlantUML for the Travel Planner Generator.
Include actors such as Tourist and Travel Agency Assistant.
Include use cases for entering preferences, generating an itinerary, viewing the website, requesting a destination image, and accessing the API.
Return only PlantUML code from @startuml to @enduml.

{uml_context}
""".strip(),
    "sequence_diagram": f"""
Generate a UML sequence diagram in valid PlantUML for the main user journey.
Show the User submitting travel preferences to the Website, the Website calling the Flask API, the Flask API calling APIFree, and the Website displaying the generated itinerary and image.
Return only PlantUML code from @startuml to @enduml.

""".strip(),
}

generated_uml = {}
for diagram_name, prompt in uml_prompts.items():
    print(f"Generating {diagram_name}...")
    uml_response = generate_text(diagram_name, prompt, project_context)
    plantuml_text = extract_plantuml(uml_response)
    generated_uml[diagram_name] = plantuml_text

    puml_path = UML_DIR / f"{diagram_name}.puml"
    png_path = UML_DIR / f"{diagram_name}.png"
    puml_path.write_text(plantuml_text, encoding="utf-8")
    render_url = render_plantuml_png(plantuml_text, png_path)

    print(f"Saved {puml_path.relative_to(PROJECT_ROOT)}")
    print(f"Saved {png_path.relative_to(PROJECT_ROOT)}")
    print(f"Rendered with {render_url}\n")

In [ ]:
for diagram_name, plantuml_text in generated_uml.items():
    print(f"--- {diagram_name} ---")
    print(plantuml_text[:800])
    print()

In [ ]:
from IPython.display import Image, display

for diagram_name in generated_uml:
    png_path = UML_DIR / f"{diagram_name}.png"
    print(f"Displaying {png_path.relative_to(PROJECT_ROOT)}")
    display(Image(filename=str(png_path)))

## Generated Flask API

This section generates a functional Flask API for the travel planner. The generated API is saved into `Task2/app`.

In [ ]:
def extract_code_block(text, language=None):
    """Extract code from a fenced markdown block, or return the raw text."""
    if language:
        pattern = rf"```{language}\s*(.*?)```"
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()

    match = re.search(r"```(?:\w+)?\s*(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


flask_api_prompt = f"""
Generate a compact Python Flask application for a travel itinerary planner.
Return only Python code, with no markdown explanation.

Required behaviour:
- Use Flask and flask_cors.CORS.
- Provide GET /health returning JSON with status ok.
- Provide GET / returning a short JSON description of the API.
- Provide POST /api/itinerary accepting JSON fields: destination, days, budget, interests, travel_style.
- Validate that destination is provided and days is an integer between 1 and 14.
- Return a JSON itinerary with destination, days, budget, interests, travel_style, overview, itinerary, and tips.
- The itinerary list should contain one item per day, each with day, morning, afternoon, evening, and budget_note.
- Support any destination string entered by the user; do not restrict the API to a fixed list of cities.
- It is acceptable to use generic travel-planning templates when real destination data is unavailable.
- Do not call external travel APIs inside the generated Flask app.
- Keep the code under 140 lines.
- Include an if __name__ == '__main__' block running on host 0.0.0.0 and port from the PORT environment variable, default 5000.

Project context:
{uml_context}
""".strip()

flask_code_response = generate_text("flask_api", flask_api_prompt, project_context)
flask_code = extract_code_block(flask_code_response, "python")

app_py_path = APP_DIR / "app.py"
app_py_path.write_text(flask_code, encoding="utf-8")
print(f"Saved {app_py_path.relative_to(PROJECT_ROOT)}")

requirements_text = """Flask==2.3.3
flask-cors==4.0.0
requests==2.31.0
"""
requirements_path = APP_DIR / "requirements.txt"
requirements_path.write_text(requirements_text, encoding="utf-8")
print(f"Saved {requirements_path.relative_to(PROJECT_ROOT)}")

In [ ]:
print(flask_code[:1200])

## Generated Website Frontend

This section follows the Week 10 practical style by generating a single `index.html` file that is served by the Flask app and integrated with the generated API.

In [ ]:
html_prompt = f"""
Generate a complete, production-ready HTML5 webpage for a Travel Planner Generator.
Return only the complete HTML document, with no markdown explanation.

Requirements:
- Use a single HTML file with inline CSS and inline JavaScript.
- Build a polished travel-planning interface, not a marketing landing page.
- Include a form with fields: destination, days, budget, interests, and travel_style.
- Use Fetch API to send POST requests to /api/itinerary as JSON.
- Display destination, overview, day-by-day itinerary, budget notes, and tips from the API response.
- Include a visible destination image area with id destinationImageArea. For now it can show a labelled placeholder that will later be replaced by the generated AI image.
- Include helpful loading, success, and error states.
- Keep all text concise and professional.
- Do not use external images or map APIs.
- The page must work when served by Flask from the same folder as app.py.

API contract:
POST /api/itinerary accepts JSON fields destination, days, budget, interests, travel_style.
The API returns JSON with destination, days, budget, interests, travel_style, overview, itinerary, and tips.

Project context:
{uml_context}
""".strip()

html_response = generate_text("website_frontend", html_prompt, project_context)
html_code = extract_code_block(html_response, "html")

index_path = APP_DIR / "index.html"
index_path.write_text(html_code, encoding="utf-8")
print(f"Saved {index_path.relative_to(PROJECT_ROOT)}")

In [ ]:
def ensure_flask_serves_index(app_path):
    """Update the generated Flask app so GET / serves index.html like the Week 10 practical."""
    code = app_path.read_text(encoding="utf-8")
    if "send_from_directory" not in code:
        code = code.replace("from flask import Flask, request, jsonify", "from flask import Flask, request, jsonify, send_from_directory")

    replacement = '''@app.route("/", methods=["GET"])
def index():
    return send_from_directory(".", "index.html")'''

    pattern = r'@app\.route\("/", methods=\["GET"\]\)\s*def index\(\):.*?(?=\n@app\.route|\nif __name__)'
    updated, count = re.subn(pattern, replacement, code, count=1, flags=re.DOTALL)
    if count == 0 and 'def index()' not in code:
        updated = code + "\n\n" + replacement + "\n"
    elif count == 0:
        updated = code

    image_route = '''

@app.route("/generated_images/<path:filename>", methods=["GET"])
def generated_image_file(filename):
    return send_from_directory(os.path.join(app.root_path, "generated_images"), filename)
'''
    if "def generated_image_file" not in updated:
        updated = updated.replace("\n@app.route(\"/api/itinerary\"", image_route + "\n@app.route(\"/api/itinerary\"")

    app_path.write_text(updated, encoding="utf-8")
    return count


route_updates = ensure_flask_serves_index(APP_DIR / "app.py")
print(f"Updated Flask index route replacements: {route_updates}")

In [ ]:
print(html_code[:1200])

## AI-Generated Destination Image

This section injects dynamic APIFree image generation into the generated Flask app. When a user submits a destination, the website requests a destination-specific generated image.

In [ ]:
import base64
import time


def inject_dynamic_image_backend(app_path):
    """Inject APIFree destination-image generation into the generated Flask app."""
    code = app_path.read_text(encoding="utf-8")
    code = code.replace("import os, logging, random, datetime", "import os, logging, random, datetime, time, base64, re")
    if "import requests" not in code:
        code = code.replace("import os, logging, random, datetime, time, base64, re", "import os, logging, random, datetime, time, base64, re\nimport requests")

    dynamic_backend = r'''
def slugify(value):
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", value.strip().lower()).strip("-")
    return slug or "destination"

def build_destination_image_prompt(destination, interests, travel_style):
    interest_text = ", ".join(interests[:4]) if interests else "local culture, food, and scenic places"
    return (
        f"Realistic editorial travel photography for a polished travel planner website: {destination}. "
        f"Show the feeling of {interest_text} with a {travel_style or 'balanced'} travel style. "
        "Bright natural light, professional composition, inviting destination atmosphere, no text, no logos, no watermark."
    )

def submit_apifree_destination_image(prompt, output_path):
    api_key = os.getenv("APIFREE_API_KEY")
    if not api_key:
        raise RuntimeError("APIFREE_API_KEY is not configured")
    base_url = os.getenv("APIFREE_BASE_URL", "https://api.apifree.ai/v1").rstrip("/")
    image_model = os.getenv("APIFREE_IMAGE_MODEL", "google/nano-banana-2")
    submit_response = requests.post(
        f"{base_url}/image/submit",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": image_model, "prompt": prompt, "negative_prompt": "blurry, distorted text, watermark, low quality, deformed objects", "width": 768, "height": 512, "num_images": 1},
        timeout=120,
    )
    submit_response.raise_for_status()
    submit_data = submit_response.json()
    if "error" in submit_data:
        raise RuntimeError(submit_data["error"].get("message", submit_data["error"]))
    request_id = submit_data.get("request_id") or submit_data.get("resp_data", {}).get("request_id")
    if not request_id:
        raise RuntimeError(f"Image submission did not return request_id: {submit_data}")
    result_url = f"{base_url}/image/{request_id}/result"
    result_data = {}
    for _ in range(24):
        result_response = requests.get(result_url, headers={"Authorization": f"Bearer {api_key}"}, timeout=120)
        result_response.raise_for_status()
        result_data = result_response.json()
        payload = result_data.get("resp_data", result_data)
        status = payload.get("status")
        if status in {"success", "completed"}:
            image_urls = payload.get("image_list") or []
            images = payload.get("images") or []
            if image_urls:
                image_response = requests.get(image_urls[0], timeout=120)
                image_response.raise_for_status()
                with open(output_path, "wb") as image_file:
                    image_file.write(image_response.content)
                return request_id
            if images and images[0].get("b64_json"):
                with open(output_path, "wb") as image_file:
                    image_file.write(base64.b64decode(images[0]["b64_json"]))
                return request_id
            if images and images[0].get("url"):
                image_response = requests.get(images[0]["url"], timeout=120)
                image_response.raise_for_status()
                with open(output_path, "wb") as image_file:
                    image_file.write(image_response.content)
                return request_id
            raise RuntimeError(f"Completed image result did not include image data: {result_data}")
        if status in {"failed", "error"}:
            raise RuntimeError(f"Image generation failed: {result_data}")
        time.sleep(5)
    raise TimeoutError(f"Image generation did not complete: {result_data}")

@app.route("/api/destination-image", methods=["POST"])
def api_destination_image():
    data = request.get_json(silent=True) or {}
    destination = (data.get("destination") or "").strip()
    if not destination:
        return jsonify({"error": "destination is required"}), 400
    interests = data.get("interests") or []
    if isinstance(interests, str):
        interests = [i.strip() for i in interests.split(",") if i.strip()]
    travel_style = data.get("travel_style") or "balanced"
    prompt = build_destination_image_prompt(destination, interests, travel_style)
    image_dir = os.path.join(app.root_path, "generated_images")
    os.makedirs(image_dir, exist_ok=True)
    filename = f"{slugify(destination)}.png"
    output_path = os.path.join(image_dir, filename)
    try:
        request_id = submit_apifree_destination_image(prompt, output_path)
        return jsonify({"destination": destination, "image_url": f"/generated_images/{filename}", "prompt": prompt, "request_id": request_id})
    except Exception as e:
        logger.exception("Error generating destination image")
        return jsonify({"error": "Image generation failed", "details": str(e)}), 500
'''

    if "def api_destination_image" not in code:
        code = code.replace("\nif __name__ == '__main__':", "\n" + dynamic_backend + "\nif __name__ == '__main__':")
    app_path.write_text(code, encoding="utf-8")


dynamic_image_doc = """# Dynamic destination image generation

The generated Flask application creates destination images at runtime. When a user submits a destination, interests, and travel style, the frontend calls `/api/destination-image`. The backend builds a fresh image prompt from those user inputs, sends it to the APIFree image API, saves the returned image in `Task2/app/generated_images/`, and returns the local image URL to the page.

This means the website is not fixed to the demonstration destination. The same app can generate a new image for Kyoto, Paris, Shanghai, New York, or another destination entered by the user.
"""
dynamic_image_doc_path = DOCS_DIR / "dynamic_image_generation.md"
dynamic_image_doc_path.write_text(dynamic_image_doc, encoding="utf-8")
print(f"Saved {dynamic_image_doc_path.relative_to(PROJECT_ROOT)}")

In [ ]:
inject_dynamic_image_backend(APP_DIR / "app.py")
print("Injected dynamic destination-image endpoint into Task2/app/app.py")

In [ ]:
def inject_dynamic_image_frontend(index_path):
    """Update the generated website so each itinerary request triggers destination-specific image generation."""
    html = index_path.read_text(encoding="utf-8")
    html = re.sub(r'<script id="generated-image-embed">.*?</script>\s*', '', html, flags=re.DOTALL)
    if "generateDestinationImage(payload);" not in html:
        html = html.replace("downloadJsonBtn.disabled = false;", "downloadJsonBtn.disabled = false;\n          generateDestinationImage(payload);")

    dynamic_function = r'''
      async function generateDestinationImage(payload){
        destinationImageArea.innerHTML = `<div style="text-align:center"><strong style="display:block">${escapeHtml(payload.destination || 'Destination')}</strong><div style="font-size:12px;color:var(--muted);margin-top:6px">Generating AI image...</div></div>`;
        try {
          const resp = await fetch('/api/destination-image', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ destination: payload.destination, interests: payload.interests, travel_style: payload.travel_style }),
            credentials: 'same-origin'
          });
          if(!resp.ok){ throw new Error('Image generation failed'); }
          const imageData = await resp.json();
          const cacheBuster = Date.now();
          destinationImageArea.innerHTML = `<img src="${imageData.image_url}?t=${cacheBuster}" alt="AI-generated image for ${escapeHtml(payload.destination)}" style="width:100%;height:100%;object-fit:cover;border-radius:8px;display:block;" />`;
        } catch(err) {
          destinationImageArea.innerHTML = `<div style="text-align:center"><strong style="display:block">${escapeHtml(payload.destination || 'Destination')}</strong><div style="font-size:12px;color:var(--muted);margin-top:6px">Image generation unavailable</div></div>`;
        }
      }
'''
    if "async function generateDestinationImage" not in html:
        html = html.replace("\n      // Download JSON", "\n" + dynamic_function + "\n      // Download JSON")
    index_path.write_text(html, encoding="utf-8")


inject_dynamic_image_frontend(APP_DIR / "index.html")
print("Injected dynamic image generation into Task2/app/index.html")

In [ ]:
print("The website now generates destination images dynamically through POST /api/destination-image.")

# Phase 3: Operation

Operation will prepare tests, Docker support, CI/CD workflow configuration, and deployment evidence for Task2.

## Automated Tests and CI/CD

This section creates lightweight pytest tests for the generated Flask API and a GitHub Actions workflow that runs the tests on every push.

In [ ]:
TESTS_DIR = TASK2_DIR / "tests"
WORKFLOW_DIR = PROJECT_ROOT / ".github" / "workflows"
TESTS_DIR.mkdir(parents=True, exist_ok=True)
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)

tests_code = r'''import sys
import types
from pathlib import Path


APP_DIR = Path(__file__).resolve().parents[1] / "app"
sys.path.insert(0, str(APP_DIR))

if "flask_cors" not in sys.modules:
    cors_module = types.ModuleType("flask_cors")
    cors_module.CORS = lambda app: app
    sys.modules["flask_cors"] = cors_module

import app as travel_app  # noqa: E402


def test_health_endpoint():
    client = travel_app.app.test_client()
    response = client.get("/health")
    assert response.status_code == 200
    assert response.get_json()["status"] == "ok"


def test_itinerary_endpoint_generates_user_destination():
    client = travel_app.app.test_client()
    response = client.post(
        "/api/itinerary",
        json={
            "destination": "Paris",
            "days": 2,
            "budget": "medium",
            "interests": ["culture", "food"],
            "travel_style": "balanced",
        },
    )
    data = response.get_json()
    assert response.status_code == 200
    assert data["destination"] == "Paris"
    assert data["days"] == 2
    assert len(data["itinerary"]) == 2
    assert "Paris" in data["overview"]


def test_itinerary_rejects_invalid_days():
    client = travel_app.app.test_client()
    response = client.post(
        "/api/itinerary",
        json={"destination": "Tokyo", "days": 30, "budget": "low"},
    )
    assert response.status_code == 400
    assert "days" in response.get_json()["error"]


def test_destination_image_endpoint_uses_dynamic_destination(monkeypatch):
    client = travel_app.app.test_client()

    def fake_submit(prompt, output_path):
        Path(output_path).write_bytes(b"fake image bytes")
        return "fake-request-id"

    monkeypatch.setattr(travel_app, "submit_apifree_destination_image", fake_submit)
    response = client.post(
        "/api/destination-image",
        json={
            "destination": "Barcelona",
            "interests": ["food", "outdoor"],
            "travel_style": "relaxed",
        },
    )
    data = response.get_json()
    assert response.status_code == 200
    assert data["destination"] == "Barcelona"
    assert "Barcelona" in data["prompt"]
    assert data["image_url"].endswith("/barcelona.png")
    assert data["request_id"] == "fake-request-id"

    generated_file = APP_DIR / "generated_images" / "barcelona.png"
    if generated_file.exists():
        generated_file.unlink()
'''

ci_workflow = '''name: Travel Planner CI

on:
  push:
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r Task2/app/requirements.txt
          pip install pytest

      - name: Run tests
        run: pytest Task2/tests -q
'''

(TESTS_DIR / "test_app.py").write_text(tests_code, encoding="utf-8")
(WORKFLOW_DIR / "ci.yml").write_text(ci_workflow, encoding="utf-8")
print("Saved Task2/tests/test_app.py")
print("Saved .github/workflows/ci.yml")

## Docker Deployment Files

This section creates Docker deployment files for the generated Flask API and website.

In [ ]:
dockerfile_text = '''FROM python:3.10-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV PORT=5000

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
'''

dockerignore_text = '''__pycache__/
*.pyc
.pytest_cache/
generated_images/
'''

deployment_notes = '''# Deployment Notes

The generated Travel Planner Generator can be deployed with Docker.

## Build

Run from `Task2/app`:

```powershell
docker build -t travel-planner-generator .
```

## Run

Run with the APIFree key supplied as an environment variable:

```powershell
docker run --rm -p 5000:5000 --env-file ../../.env travel-planner-generator
```

Then open:

```text
http://localhost:5000
```

## Evidence

For the coursework deployment screenshot, capture the running website in the browser at `http://localhost:5000`.
'''

(APP_DIR / "Dockerfile").write_text(dockerfile_text, encoding="utf-8")
(APP_DIR / ".dockerignore").write_text(dockerignore_text, encoding="utf-8")
(TASK2_DIR / "deployment.md").write_text(deployment_notes, encoding="utf-8")
print("Saved Task2/app/Dockerfile")
print("Saved Task2/app/.dockerignore")
print("Saved Task2/deployment.md")